In [1]:
import sys
sys.path.append('../cmldask')
import os, pickle, warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
from statsmodels.stats.multitest import multipletests
from cmldask import CMLDask as da
from dask.distributed import wait, as_completed
import gc
import itertools

from ptsa.data.filters import ButterworthFilter, MorletWaveletFilter
import cmlreaders as cml
from compute_scalp_features import create_baseline_events

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option('display.max_columns', None)

In [2]:
EXP = 'CourierReinstate1'
# SUBJECTS = ['LTP564', 'LTP565']
SUBJECTS = ['LTP564', 'LTP565', 'LTP566', 'LTP567', 'LTP568', 'LTP569', 'LTP571', 'LTP572', 'LTP573',
            'LTP574', 'LTP575', 'LTP576', 'LTP577', 'LTP578', 'LTP579', 'LTP580', 'LTP581', 'LTP583',
            'LTP584', 'LTP585', 'LTP586', 'LTP587', 'LTP588', 'LTP589', 'LTP590', 'LTP591', 'LTP592', 
            'LTP593', 'LTP594', 'LTP595', 'LTP596', 'LTP597', 'LTP598', 'LTP599', 'LTP600', 'LTP601', 
            'LTP602', 'LTP603', 'LTP604', 'LTP605']

RETR_REL_START = -900
RETR_REL_STOP  =  -50
DELIB_REL_START = -900
DELIB_REL_STOP  = -50
BUFFER_MS_DELIB = 833.333333333

WIDTH = 6 
FREQS = np.logspace(np.log10(2), np.log10(100), 46)
NOTCH_BAND = (58., 62.)
BATCH_EVENTS = 64
ROI_ORDER = ['LAI','RAI','LAS','RAS','LPS','RPS','LPI','RPI']

In [3]:
def assign_roi(channel):
    roi_dict = {
        'LAS':['C24','C25','D2','D3','D4','D11','D12','D13'],
        'LAI':['C31','C32','D5','D6','D9','D10','D21','D22'],
        'LPS':['D29','A5','A6','A7','A8','A17','A18'],
        'LPI':['D30','D31','A9','A10','A11','A15','A16'],
        'RAS':['B30','B31','B32','C2','C3','C4','C11','C12'],
        'RAI':['B24','B25','B28','B29','C5','C6','C9','C10'],
        'RPS':['A30','A31','A32','B3','B4','B5','B13'],
        'RPI':['A28','A29','B6','B7','B8','B11','B12'],
    }
    for roi, chans in roi_dict.items():
        if channel in chans:
            return roi
    return None

In [4]:
df_idx = cml.get_data_index('ltp').query("experiment == @EXP").copy()
if SUBJECTS is not None:
    df_idx = df_idx[df_idx['subject'].isin(SUBJECTS)]
subjects = sorted(df_idx.subject.unique())
print(f'Found {len(subjects)} subjects for experiment "{EXP}".')

Found 40 subjects for experiment "CourierReinstate1".


In [5]:
REQUIRED_NUMERIC = ['session', 'trial', 'mstime']
OPTIONAL_NUMERIC = ['intrusion', 'serialpos', 'repetition', 'itemno', 'list', 'eegoffset']
STRING_COLS      = ['type', 'eegfile', 'phase']

def prep_for_baseline(evs: pd.DataFrame) -> pd.DataFrame:
    
    # Ensure trial column
    if 'trial' not in evs.columns:
        for alt in ['list','trialno','trial_num','trial_number']:
            if alt in evs.columns:
                evs = evs.rename(columns={alt: 'trial'})
                break

    evs = evs.replace(r'^\s*$', np.nan, regex=True)
    
    # Coerce numeric columns, replace NaN with 0
    for col in REQUIRED_NUMERIC + OPTIONAL_NUMERIC:
        if col in evs.columns:
            evs[col] = pd.to_numeric(evs[col], errors='coerce')
    if 'intrusion' in evs.columns:
        evs['intrusion'] = evs['intrusion'].fillna(0)

    # Ensure anchors exist and drop rows missing anchors
    if not set(REQUIRED_NUMERIC).issubset(evs.columns):
        return evs.iloc[0:0].copy()
    evs = evs.dropna(subset=REQUIRED_NUMERIC)
    
    # Cast anchors to int64
    for col in REQUIRED_NUMERIC:
        evs[col] = evs[col].astype(np.int64)

    # Ensure required string columns exist and are string dtype
    for col in STRING_COLS:
        if col not in evs.columns:
            evs[col] = ''
        else:
            evs[col] = evs[col].fillna('').astype(str)

    return evs

def df_to_recarray_with_dtype(df: pd.DataFrame) -> np.recarray:
    """
    Build a structured recarray with explicit dtypes:
      - REQUIRED_NUMERIC & OPTIONALLY numeric columns: int64/float64
      - STRING_COLS ('type','eegfile','phase'): Unicode strings
      - others: use a conservative string fallback
    """
    names, formats, arrays = [], [], []

    for name in df.columns:
        if name in STRING_COLS:
            arr = df[name].fillna('').astype(str).to_numpy()
            fmt = 'U128'
        else:
            dt = df[name].dtype
            if np.issubdtype(dt, np.integer):
                arr = df[name].astype(np.int64).to_numpy()
                fmt = np.int64
            elif np.issubdtype(dt, np.floating):
                arr = df[name].astype(np.float64).to_numpy()
                fmt = np.float64
            else:
                arr = df[name].fillna('').astype(str).to_numpy()
                fmt = 'U128'
        names.append(name); formats.append(fmt); arrays.append(arr)

    rec = np.rec.fromarrays(arrays, names=names, formats=formats)
    return rec

In [6]:
def process_subject_stats(subject, EXP=EXP, batch=BATCH_EVENTS):
    """
    Returns DataFrame: subject, session, trial, type, outputpos, serialpos, item, freq, power
    One row = one REC_WORD (retrieval) or REC_BASE (matched deliberation) event at one freq
    Power averaged across channels
    """
    sessions = cml.get_data_index('ltp').query("experiment == @EXP and subject == @subject").session.unique()

    dfs = []
    freq_vec = np.array(FREQS)

    for sess in sessions:
        try:
            reader = cml.CMLReader(subject=subject, experiment=EXP, session=sess)
            evs_all = reader.load('task_events')
            if evs_all.empty:
                continue
                
            has_phase = 'phase' in evs_all.columns
            rec_mask = evs_all['type'].isin(['REC_START', 'REC_STOP', 'REC_WORD', 'REC_WORD_VV'])
            if has_phase:
                rec_mask &= (evs_all['phase'] != 'practice')
            rec_evs0 = evs_all.loc[rec_mask].reset_index(drop=True)

            if rec_evs0.empty or (rec_evs0['type'].eq('REC_START').sum()==0) or (rec_evs0['type'].eq('REC_STOP').sum()==0):
                print(f"[{subject} s{sess}] no recall bounds -> skip")
                continue

            rec_evs0 = prep_for_baseline(rec_evs0)
            if rec_evs0.empty or (rec_evs0['type'].eq('REC_WORD').sum() == 0):
                print(f"[{subject} s{sess}] cleaned table empty -> skip")
                continue
                
            events_rec = df_to_recarray_with_dtype(rec_evs0)

            rec_evs = pd.DataFrame.from_records(
                create_baseline_events(events_rec, start_time=3000, end_time=90000)
            ).reset_index(drop=True)

            pairs = rec_evs.query("type in ['REC_WORD', 'REC_BASE']")
            if pairs.empty:
                print(f"[{subject} s{sess}] no REC_WORD/REC_BASE -> skip")
                continue

            # remove repetitions
            if 'item' in pairs.columns:
                pairs = pairs.sort_values(['session', 'trial', 'mstime'])
                pairs['repetition'] = 0
                seen = {}
                for i, row in pairs.iterrows():
                    if row['type'] != 'REC_WORD':
                        continue
                    key = (row['session'], row['trial'])
                    if key not in seen:
                        seen[key] = set()
                    if row['item'] in seen[key]:
                        pairs.at[i, 'repetition'] = 1
                    else:
                        seen[key].add(row['item'])
                pairs = pairs.query("(type == 'REC_BASE') | (repetition == 0)")
                pairs.drop(columns=['repetition'], inplace=True, errors='ignore')

            if 'intrusion' in pairs.columns:  # drop intrusions for REC_WORD
                pairs = pairs.query("intrusion == 0 or type == 'REC_BASE'")
            if 'eegoffset' in pairs.columns:
                pairs = pairs[pairs['eegoffset'].fillna(0) >= 0]
            if set(['eegfile', 'eegoffset']).issubset(pairs.columns):
                pairs = pairs.drop_duplicates(subset=['eegfile', 'eegoffset'])

            pairs = pairs.sort_values(['trial', 'mstime']).reset_index(drop=True)
            pairs['outputpos'] = np.nan
            rec_word_mask = pairs['type'] == 'REC_WORD'
            pairs.loc[rec_word_mask, 'outputpos'] = pairs.loc[rec_word_mask].groupby('trial').cumcount() + 1

            for start in range(0, len(pairs), batch):
                evs = pairs.iloc[start:start+batch]
                eeg = reader.load_eeg(
                    evs,
                    rel_start=RETR_REL_START - BUFFER_MS_DELIB,
                    rel_stop=RETR_REL_STOP + BUFFER_MS_DELIB,
                    clean='LCF'
                ).to_ptsa()

                # Notch filter
                eeg = ButterworthFilter(freq_range=list(NOTCH_BAND), filt_type='stop', order=4).filter(eeg)

                # Wavelet power
                mw = MorletWaveletFilter(freqs=freq_vec, width=WIDTH, output='power', complete=True)
                pwr = mw.filter(eeg).sel(time=slice(RETR_REL_START, RETR_REL_STOP)).mean(dim='time')
                p = pwr.transpose('event', 'channel', 'frequency').values
                p_event_freq = p.mean(axis=1)  # averaged across channels
                
                # Event-level metadata
                meta_cols = ['subject', 'session', 'trial', 'type', 'outputpos', 'serialpos', 'item']
                meta = evs[meta_cols]
                
                power_df = pd.DataFrame(p_event_freq, columns=freq_vec, index=meta.index)
                batch_df = pd.concat([meta.reset_index(drop=True), power_df.reset_index(drop=True)], axis=1)
                batch_long = batch_df.melt(id_vars=meta_cols, var_name='freq', value_name='power')
                dfs.append(batch_long)

                del eeg, pwr, p, p_event_freq, power_df, batch_df, batch_long
                gc.collect()

        except Exception as e:
            print(f"[WARN] {subject} session {sess} skipped: {e}")
            continue

    if not dfs:
        return pd.DataFrame(columns=['subject', 'session', 'trial', 'type', 'outputpos', 'serialpos', 'item', 'freq', 'power'])
    
    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values(['subject', 'session', 'trial', 'type', 'outputpos', 'serialpos', 'freq']).reset_index(drop=True)
    
    return df

In [7]:
client = da.new_dask_client_slurm(
    job_name="retrieval",
    memory_per_job="50GB",
    max_n_jobs=10,
    queue="RAM",
    local_directory="",
    log_directory="dask_logs"
)

Unique port for joycerose14 is 51609
{'dashboard_address': ':51609'}
To view the dashboard, run: 
`ssh -fN joycerose14@rhino2.psych.upenn.edu -L 8000:192.168.86.104:51609` in your local computer's terminal (NOT rhino) 
and then navigate to localhost:8000 in your browser


In [8]:
futures = [client.submit(process_subject_stats, s, EXP, BATCH_EVENTS) for s in subjects]

In [9]:
all_dfs, n_done, n_fail = [], 0, 0
for fut in as_completed(futures):
    try:
        df_sub = fut.result()
        all_dfs.append(df_sub)
        n_done += 1
        sub = df_sub['subject'].iat[0] if not df_sub.empty else 'EMPTY_SUBJECT'
        print(f"[DONE] {n_done}/{len(subjects)} :: {sub} (rows={len(df_sub)})")
    except Exception as e:
        n_fail += 1
        print(f"[ERR] Future failed ({n_fail} fails): {e}")

[DONE] 1/40 :: LTP565 (rows=16284)
[DONE] 2/40 :: LTP569 (rows=9338)
[DONE] 3/40 :: LTP564 (rows=6118)
[DONE] 4/40 :: LTP567 (rows=33166)
[DONE] 5/40 :: LTP580 (rows=29716)
[DONE] 6/40 :: EMPTY_SUBJECT (rows=0)
[DONE] 7/40 :: LTP585 (rows=42412)
[DONE] 8/40 :: LTP578 (rows=32016)
[DONE] 9/40 :: LTP583 (rows=34408)
[DONE] 10/40 :: LTP572 (rows=35604)
[DONE] 11/40 :: LTP566 (rows=12420)
[DONE] 12/40 :: LTP568 (rows=25438)
[DONE] 13/40 :: LTP577 (rows=10166)
[DONE] 14/40 :: LTP574 (rows=44160)
[DONE] 15/40 :: LTP576 (rows=48438)
[DONE] 16/40 :: LTP590 (rows=18446)
[DONE] 17/40 :: LTP593 (rows=9706)
[DONE] 18/40 :: LTP586 (rows=44712)
[DONE] 19/40 :: LTP584 (rows=42412)
[DONE] 20/40 :: LTP589 (rows=34730)
[DONE] 21/40 :: LTP581 (rows=35098)
[DONE] 22/40 :: LTP592 (rows=13984)
[DONE] 23/40 :: LTP575 (rows=26312)
[DONE] 24/40 :: LTP587 (rows=43010)
[DONE] 25/40 :: LTP600 (rows=3864)
[DONE] 26/40 :: LTP596 (rows=12558)
[DONE] 27/40 :: LTP573 (rows=37030)
[DONE] 28/40 :: LTP571 (rows=46644)
[D

In [11]:
client.shutdown()

In [12]:
df_all = pd.concat(all_dfs, ignore_index=True)
df_all

,subject,session,trial,type,outputpos,serialpos,item,freq,power
0,LTP565,3,0,REC_BASE,NaN,0,,2.0,3.326119e-05
1,LTP565,3,0,REC_BASE,NaN,0,,2.0,2.492266e-05
2,LTP565,3,0,REC_BASE,NaN,0,,2.0,3.275833e-05
3,LTP565,3,0,REC_BASE,NaN,0,,2.0,5.268584e-05
4,LTP565,3,0,REC_BASE,NaN,0,,2.0,8.550346e-05
...,...,...,...,...,...,...,...,...,...
1140657,LTP603,5,9,REC_WORD,3.0,7,FOX,70.628575,7.732547e-07
1140658,LTP603,5,9,REC_WORD,3.0,7,FOX,77.043381,7.614654e-07
1140659,LTP603,5,9,REC_WORD,3.0,7,FOX,84.040809,6.816916e-07
1140660,LTP603,5,9,REC_WORD,3.0,7,FOX,91.673774,5.757031e-07


In [13]:
df_all.to_csv('csv/retrieval_df.csv', index=False)